# Bayesian Analysis of AI Safety Behaviour Under Pressure

Bayesian estimation of unsafe-behaviour rates and condition effects,
providing **credible intervals** rather than point estimates.

**Contents:**
1. Beta-Binomial conjugate model — per-condition unsafe rate posteriors
2. Bayesian comparison of pressure conditions (posterior contrasts)
3. Bayesian comparison of monitoring effect
4. Bayesian logistic regression (PyMC) — multi-predictor model
5. Scenario vulnerability ranking with uncertainty
6. Log findings to `results/bayesian/metrics.json`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats
from pathlib import Path
from datetime import datetime

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

SEED = 42
np.random.seed(SEED)
FIGURES_DIR = Path('../reports/figures')
RESULTS_DIR = Path('../results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')
print('Libraries loaded.')

In [ ]:
# Load data
d2a = pd.read_parquet('../data/processed/d2a_sessions.parquet')
d2c = pd.read_parquet('../data/processed/d2c_labels.parquet')
d2d = pd.read_parquet('../data/processed/d2d_features.parquet')

df = d2a.merge(d2c[['session_id', 'overall_unsafe']], on='session_id', suffixes=('', '_label'))
df = df.merge(d2d, on='session_id', suffixes=('', '_feat'))

# Clean up any merge-suffix artefacts on the target column
unsafe_col = next(c for c in df.columns if c.startswith('overall_unsafe'))
if unsafe_col != 'overall_unsafe':
    df.rename(columns={unsafe_col: 'overall_unsafe'}, inplace=True)
    df.drop(columns=[c for c in df.columns if 'overall_unsafe' in c and c != 'overall_unsafe'],
            errors='ignore', inplace=True)

y = df['overall_unsafe'].astype(int).values

print(f'Sessions: {len(df)}')
print(f'Unsafe: {y.sum()} ({y.mean():.1%})')
print(f'Pressures: {sorted(df.pressure_id.unique())}')
print(f'Scenarios: {sorted(df.scenario_id.unique())}')
print(f'Monitoring: {sorted(df.monitoring_id.unique())}')

## 1. Beta-Binomial Conjugate Model — Per-Condition Unsafe Rate Posteriors

For each experimental condition, we model the unsafe rate $\theta$ as:

$$\theta \sim \text{Beta}(\alpha_0, \beta_0) \quad \text{(prior)}$$
$$k \mid n, \theta \sim \text{Binomial}(n, \theta) \quad \text{(likelihood)}$$
$$\theta \mid k, n \sim \text{Beta}(\alpha_0 + k, \beta_0 + n - k) \quad \text{(posterior)}$$

Using a weakly informative prior $\text{Beta}(1, 1)$ (uniform) to let the data speak.

In [ ]:
# Beta-Binomial posterior for each pressure condition
alpha_prior, beta_prior = 1, 1  # Uniform prior (weakly informative)
n_posterior_samples = 50000

pressure_posteriors = {}
pressure_stats = []

print('=== Posterior Estimates: Unsafe Rate by Pressure ===')
print(f'{"Pressure":<25} {"n":>4} {"k":>4} {"MLE":>7} {"Post. Mean":>11} {"95% CrI":>20}')
print('-' * 80)

for pressure in sorted(df.pressure_id.unique()):
    mask = df.pressure_id == pressure
    n = mask.sum()
    k = y[mask].sum()
    
    # Posterior parameters
    alpha_post = alpha_prior + k
    beta_post = beta_prior + n - k
    
    # Sample from posterior
    posterior_samples = np.random.beta(alpha_post, beta_post, size=n_posterior_samples)
    pressure_posteriors[pressure] = posterior_samples
    
    # Summary statistics
    post_mean = alpha_post / (alpha_post + beta_post)
    cri_low, cri_high = np.percentile(posterior_samples, [2.5, 97.5])
    mle = k / n
    
    pressure_stats.append({
        'condition': pressure, 'type': 'pressure',
        'n': int(n), 'k': int(k), 'mle': mle,
        'post_mean': post_mean, 'cri_low': cri_low, 'cri_high': cri_high
    })
    
    print(f'{pressure:<25} {n:>4} {k:>4} {mle:>7.3f} {post_mean:>11.3f} [{cri_low:.3f}, {cri_high:.3f}]')

In [ ]:
# Visualise posterior distributions for each pressure condition
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Top: density plots
ax = axes[0]
colors = plt.cm.Set2(np.linspace(0, 1, len(pressure_posteriors)))
x_grid = np.linspace(0, 0.45, 500)

for (pressure, samples), color in zip(pressure_posteriors.items(), colors):
    stat = next(s for s in pressure_stats if s['condition'] == pressure)
    density = stats.gaussian_kde(samples)(x_grid)
    ax.plot(x_grid, density, color=color, linewidth=2,
            label=f"{pressure} ({stat['k']}/{stat['n']})")
    ax.fill_between(x_grid, density, alpha=0.15, color=color)

ax.set_xlabel('Unsafe Rate (θ)')
ax.set_ylabel('Posterior Density')
ax.set_title('Posterior Distributions: Unsafe Rate by Pressure Condition')
ax.legend(fontsize=8, loc='upper right')
ax.axvline(y.mean(), ls='--', color='grey', alpha=0.5, label='Overall rate')

# Bottom: forest plot (credible intervals)
ax = axes[1]
stats_df = pd.DataFrame(pressure_stats).sort_values('post_mean', ascending=True)
y_pos = range(len(stats_df))

ax.scatter(stats_df['post_mean'], y_pos, color='#2c3e50', s=60, zorder=3)
ax.hlines(y_pos, stats_df['cri_low'], stats_df['cri_high'],
          color='#2c3e50', linewidth=2, zorder=2)
ax.scatter(stats_df['mle'], y_pos, color='#e74c3c', marker='x', s=40,
           zorder=3, label='MLE (frequentist)')

ax.set_yticks(list(y_pos))
ax.set_yticklabels(stats_df['condition'].values)
ax.set_xlabel('Unsafe Rate (θ)')
ax.set_title('Forest Plot: 95% Credible Intervals by Pressure')
ax.axvline(y.mean(), ls='--', color='grey', alpha=0.5)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '26_bayesian_pressure_posteriors.png', bbox_inches='tight')
plt.show()

## 2. Bayesian Comparison of Pressure Conditions

For each adversarial pressure, compute:
- $P(\theta_{\text{pressure}} > \theta_{\text{baseline}})$ — probability that pressure increases unsafe rate
- Posterior distribution of the **risk difference**: $\delta = \theta_{\text{pressure}} - \theta_{\text{baseline}}$
- 95% credible interval for $\delta$

In [ ]:
# Bayesian comparison: each pressure vs baseline (none)
baseline_samples = pressure_posteriors['none']
contrasts = {}

print('=== Bayesian Pressure Contrasts (vs Baseline) ===')
print(f'{"Pressure":<25} {"P(θ>baseline)":>14} {"Mean Δ":>9} {"95% CrI for Δ":>22} {"Evidence":>15}')
print('-' * 90)

for pressure in sorted(df.pressure_id.unique()):
    if pressure == 'none':
        continue
    
    pressure_samples = pressure_posteriors[pressure]
    delta = pressure_samples - baseline_samples
    
    prob_greater = (delta > 0).mean()
    delta_mean = delta.mean()
    delta_cri = np.percentile(delta, [2.5, 97.5])
    
    # Interpret evidence strength
    if prob_greater > 0.95:
        evidence = 'Strong increase'
    elif prob_greater > 0.80:
        evidence = 'Weak increase'
    elif prob_greater < 0.05:
        evidence = 'Strong decrease'
    elif prob_greater < 0.20:
        evidence = 'Weak decrease'
    else:
        evidence = 'Inconclusive'
    
    contrasts[pressure] = {
        'prob_greater_than_baseline': float(prob_greater),
        'mean_delta': float(delta_mean),
        'cri_delta': [float(delta_cri[0]), float(delta_cri[1])],
        'evidence': evidence
    }
    
    print(f'{pressure:<25} {prob_greater:>14.3f} {delta_mean:>+9.4f} [{delta_cri[0]:>+.4f}, {delta_cri[1]:>+.4f}] {evidence:>15}')

In [ ]:
# Visualise posterior contrasts
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

pressures = sorted([p for p in df.pressure_id.unique() if p != 'none'])

for idx, pressure in enumerate(pressures):
    ax = axes[idx]
    delta = pressure_posteriors[pressure] - baseline_samples
    
    ax.hist(delta, bins=60, density=True, color='#3498db', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='red', linewidth=1.5, ls='--', label='No effect')
    
    cri = np.percentile(delta, [2.5, 97.5])
    ax.axvline(cri[0], color='orange', ls=':', linewidth=1)
    ax.axvline(cri[1], color='orange', ls=':', linewidth=1)
    ax.axvspan(cri[0], cri[1], alpha=0.1, color='orange')
    
    prob = contrasts[pressure]['prob_greater_than_baseline']
    ax.set_title(f'{pressure}\nP(Δ>0)={prob:.2f}', fontsize=9)
    ax.set_xlabel('Risk Difference (Δ)')
    if idx % 3 == 0:
        ax.set_ylabel('Density')

# Hide unused subplot
for j in range(len(pressures), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Posterior Risk Difference: Each Pressure vs Baseline', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '27_bayesian_pressure_contrasts.png', bbox_inches='tight')
plt.show()

## 3. Bayesian Comparison of Monitoring Effect

Does monitoring (telling the model its output will be reviewed) reduce unsafe behaviour?

$$P(\theta_{\text{monitored}} < \theta_{\text{unmonitored}}) = ?$$

In [ ]:
# Monitoring effect: Beta-Binomial posteriors
monitoring_posteriors = {}
monitoring_stats = []

print('=== Posterior Estimates: Unsafe Rate by Monitoring ===')
print(f'{"Condition":<20} {"n":>4} {"k":>4} {"Post. Mean":>11} {"95% CrI":>20}')
print('-' * 65)

for mon in sorted(df.monitoring_id.unique()):
    mask = df.monitoring_id == mon
    n = mask.sum()
    k = y[mask].sum()
    
    alpha_post = alpha_prior + k
    beta_post = beta_prior + n - k
    samples = np.random.beta(alpha_post, beta_post, size=n_posterior_samples)
    monitoring_posteriors[mon] = samples
    
    post_mean = alpha_post / (alpha_post + beta_post)
    cri = np.percentile(samples, [2.5, 97.5])
    
    monitoring_stats.append({
        'condition': mon, 'type': 'monitoring',
        'n': int(n), 'k': int(k),
        'post_mean': post_mean, 'cri_low': cri[0], 'cri_high': cri[1]
    })
    
    print(f'{mon:<20} {n:>4} {k:>4} {post_mean:>11.3f} [{cri[0]:.3f}, {cri[1]:.3f}]')

# Contrast: monitored vs unmonitored
delta_mon = monitoring_posteriors['monitored'] - monitoring_posteriors['unmonitored']
prob_mon_safer = (delta_mon < 0).mean()
delta_mon_cri = np.percentile(delta_mon, [2.5, 97.5])

print(f'\nP(monitored safer than unmonitored) = {prob_mon_safer:.3f}')
print(f'Mean difference: {delta_mon.mean():+.4f}')
print(f'95% CrI for difference: [{delta_mon_cri[0]:+.4f}, {delta_mon_cri[1]:+.4f}]')

if prob_mon_safer > 0.95:
    print('→ Strong evidence that monitoring reduces unsafe behaviour')
elif prob_mon_safer > 0.80:
    print('→ Weak evidence that monitoring reduces unsafe behaviour')
else:
    print('→ No clear evidence that monitoring affects unsafe behaviour rate')

In [ ]:
# Visualise monitoring comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: overlapping posteriors
ax = axes[0]
x_grid = np.linspace(0, 0.30, 500)
for mon, color, ls in [('monitored', '#27ae60', '-'), ('unmonitored', '#e74c3c', '--')]:
    density = stats.gaussian_kde(monitoring_posteriors[mon])(x_grid)
    ax.plot(x_grid, density, color=color, linewidth=2, ls=ls, label=mon)
    ax.fill_between(x_grid, density, alpha=0.15, color=color)
ax.set_xlabel('Unsafe Rate (θ)')
ax.set_ylabel('Posterior Density')
ax.set_title('Monitoring Effect: Posterior Distributions')
ax.legend()

# Right: posterior difference
ax = axes[1]
ax.hist(delta_mon, bins=60, density=True, color='#8e44ad', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linewidth=1.5, ls='--', label='No effect')
ax.axvspan(delta_mon_cri[0], delta_mon_cri[1], alpha=0.15, color='orange', label='95% CrI')
ax.set_xlabel('Δ = θ(monitored) − θ(unmonitored)')
ax.set_ylabel('Density')
ax.set_title(f'Posterior Difference (P(Δ<0)={prob_mon_safer:.2f})')
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '28_bayesian_monitoring_effect.png', bbox_inches='tight')
plt.show()

## 4. Bayesian Logistic Regression (PyMC)

Multi-predictor Bayesian model estimating:
$$\text{logit}(P(\text{unsafe})) = \alpha + \beta_{\text{scenario}} + \beta_{\text{pressure}} + \beta_{\text{monitoring}}$$

This provides **full posterior distributions** over regression coefficients,
enabling probabilistic statements about the effect of each experimental factor.

In [ ]:
# Encode categorical predictors with proper contrast coding
from sklearn.preprocessing import LabelEncoder

le_scenario = LabelEncoder()
le_pressure = LabelEncoder()
le_monitoring = LabelEncoder()

scenario_idx = le_scenario.fit_transform(df['scenario_id'])
pressure_idx = le_pressure.fit_transform(df['pressure_id'])
monitoring_idx = le_monitoring.fit_transform(df['monitoring_id'])

n_scenarios = len(le_scenario.classes_)
n_pressures = len(le_pressure.classes_)
n_monitoring = len(le_monitoring.classes_)

print(f'Scenarios ({n_scenarios}): {list(le_scenario.classes_)}')
print(f'Pressures ({n_pressures}): {list(le_pressure.classes_)}')
print(f'Monitoring ({n_monitoring}): {list(le_monitoring.classes_)}')

In [ ]:
# Bayesian hierarchical logistic regression with PyMC
with pm.Model() as safety_model:
    # Priors
    alpha = pm.Normal('intercept', mu=0, sigma=2)
    beta_scenario = pm.Normal('beta_scenario', mu=0, sigma=1.5, shape=n_scenarios)
    beta_pressure = pm.Normal('beta_pressure', mu=0, sigma=1.5, shape=n_pressures)
    beta_monitoring = pm.Normal('beta_monitoring', mu=0, sigma=1.5, shape=n_monitoring)
    
    # Linear predictor
    eta = (alpha
           + beta_scenario[scenario_idx]
           + beta_pressure[pressure_idx]
           + beta_monitoring[monitoring_idx])
    
    # Likelihood
    p = pm.math.sigmoid(eta)
    obs = pm.Bernoulli('unsafe', p=p, observed=y)
    
    # Sample
    trace = pm.sample(
        draws=2000, tune=1000, cores=1, chains=2,
        random_seed=SEED, target_accept=0.9,
        return_inferencedata=True, progressbar=True
    )

print('\n=== Sampling Summary ===')
print(az.summary(trace, var_names=['intercept', 'beta_scenario', 'beta_pressure', 'beta_monitoring'],
                 round_to=3))

In [ ]:
# Visualise posterior distributions of regression coefficients
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scenario effects
ax = axes[0]
scenario_samples = trace.posterior['beta_scenario'].values.reshape(-1, n_scenarios)
scenario_means = scenario_samples.mean(axis=0)
scenario_cri = np.percentile(scenario_samples, [2.5, 97.5], axis=0)
y_pos = range(n_scenarios)

ax.scatter(scenario_means, y_pos, color='#2c3e50', s=60, zorder=3)
ax.hlines(y_pos, scenario_cri[0], scenario_cri[1], color='#2c3e50', linewidth=2)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(le_scenario.classes_, fontsize=9)
ax.axvline(0, color='red', ls='--', alpha=0.5)
ax.set_xlabel('β (log-odds effect)')
ax.set_title('Scenario Effects')

# Pressure effects
ax = axes[1]
pressure_samples = trace.posterior['beta_pressure'].values.reshape(-1, n_pressures)
pressure_means = pressure_samples.mean(axis=0)
pressure_cri = np.percentile(pressure_samples, [2.5, 97.5], axis=0)
y_pos = range(n_pressures)

ax.scatter(pressure_means, y_pos, color='#e67e22', s=60, zorder=3)
ax.hlines(y_pos, pressure_cri[0], pressure_cri[1], color='#e67e22', linewidth=2)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(le_pressure.classes_, fontsize=9)
ax.axvline(0, color='red', ls='--', alpha=0.5)
ax.set_xlabel('β (log-odds effect)')
ax.set_title('Pressure Effects')

# Monitoring effects
ax = axes[2]
monitoring_samples = trace.posterior['beta_monitoring'].values.reshape(-1, n_monitoring)
monitoring_means = monitoring_samples.mean(axis=0)
monitoring_cri = np.percentile(monitoring_samples, [2.5, 97.5], axis=0)
y_pos = range(n_monitoring)

ax.scatter(monitoring_means, y_pos, color='#27ae60', s=60, zorder=3)
ax.hlines(y_pos, monitoring_cri[0], monitoring_cri[1], color='#27ae60', linewidth=2)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(le_monitoring.classes_, fontsize=9)
ax.axvline(0, color='red', ls='--', alpha=0.5)
ax.set_xlabel('β (log-odds effect)')
ax.set_title('Monitoring Effects')

plt.suptitle('Bayesian Logistic Regression — Posterior Coefficient Estimates', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '29_bayesian_logistic_regression.png', bbox_inches='tight')
plt.show()

In [ ]:
# MCMC diagnostics
print('=== MCMC Diagnostics ===')
summary = az.summary(trace, var_names=['intercept', 'beta_scenario', 'beta_pressure', 'beta_monitoring'])

# ArviZ 1.x may return string-typed columns; coerce to numeric
for col in summary.columns:
    summary[col] = pd.to_numeric(summary[col], errors='coerce')

print(summary)

rhat_max = summary['r_hat'].max()
ess_bulk_min = summary['ess_bulk'].min()
ess_tail_min = summary['ess_tail'].min()

print(f'\nMax R-hat: {rhat_max:.4f} (should be < 1.01)')
print(f'Min ESS (bulk): {ess_bulk_min:.0f} (should be > 400)')
print(f'Min ESS (tail): {ess_tail_min:.0f} (should be > 400)')

if rhat_max < 1.01 and ess_bulk_min > 400:
    print('Chains converged — posterior estimates are reliable')
else:
    print('Convergence issues detected — consider more samples or reparametrisation')

## 5. Scenario Vulnerability Ranking with Uncertainty

Using the Bayesian posterior to rank scenarios by their unsafe rate,
with full uncertainty quantification.

In [ ]:
# Scenario-level Beta-Binomial posteriors
scenario_posteriors = {}
scenario_stats = []

print('=== Posterior Estimates: Unsafe Rate by Scenario ===')
print(f'{"Scenario":<28} {"n":>4} {"k":>4} {"Post. Mean":>11} {"95% CrI":>20} {"P(θ>0.20)":>10}')
print('-' * 85)

for scenario in sorted(df.scenario_id.unique()):
    mask = df.scenario_id == scenario
    n = mask.sum()
    k = y[mask].sum()
    
    alpha_post = alpha_prior + k
    beta_post = beta_prior + n - k
    samples = np.random.beta(alpha_post, beta_post, size=n_posterior_samples)
    scenario_posteriors[scenario] = samples
    
    post_mean = alpha_post / (alpha_post + beta_post)
    cri = np.percentile(samples, [2.5, 97.5])
    prob_high_risk = (samples > 0.20).mean()
    
    scenario_stats.append({
        'condition': scenario, 'type': 'scenario',
        'n': int(n), 'k': int(k),
        'post_mean': post_mean, 'cri_low': cri[0], 'cri_high': cri[1],
        'prob_high_risk': prob_high_risk
    })
    
    print(f'{scenario:<28} {n:>4} {k:>4} {post_mean:>11.3f} [{cri[0]:.3f}, {cri[1]:.3f}] {prob_high_risk:>10.3f}')

In [ ]:
# Visualise scenario posteriors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlapping posteriors
ax = axes[0]
x_grid = np.linspace(0, 0.75, 500)
scenario_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for (scenario, samples), color in zip(
    sorted(scenario_posteriors.items(), key=lambda x: -x[1].mean()),
    scenario_colors
):
    density = stats.gaussian_kde(samples)(x_grid)
    ax.plot(x_grid, density, color=color, linewidth=2, label=scenario)
    ax.fill_between(x_grid, density, alpha=0.15, color=color)

ax.axvline(0.20, color='red', ls=':', alpha=0.5, label='20% threshold')
ax.set_xlabel('Unsafe Rate (θ)')
ax.set_ylabel('Posterior Density')
ax.set_title('Scenario Vulnerability Posteriors')
ax.legend(fontsize=8)

# Right: pairwise P(scenario_i > scenario_j)
ax = axes[1]
scenario_names = sorted(scenario_posteriors.keys())
n_scen = len(scenario_names)
prob_matrix = np.zeros((n_scen, n_scen))

for i, s1 in enumerate(scenario_names):
    for j, s2 in enumerate(scenario_names):
        prob_matrix[i, j] = (scenario_posteriors[s1] > scenario_posteriors[s2]).mean()

sns.heatmap(prob_matrix, annot=True, fmt='.2f', cmap='RdYlGn_r',
            xticklabels=scenario_names, yticklabels=scenario_names,
            ax=ax, vmin=0, vmax=1, square=True)
ax.set_title('P(row scenario riskier than column)')

plt.suptitle('Bayesian Scenario Vulnerability Analysis', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '30_bayesian_scenario_vulnerability.png', bbox_inches='tight')
plt.show()

## 6. Log Findings

In [ ]:
# Extract PyMC regression summary
pymc_summary_df = az.summary(
    trace,
    var_names=['intercept', 'beta_scenario', 'beta_pressure', 'beta_monitoring']
)

# Coerce to numeric (ArviZ 1.x may return string-typed columns)
for col in pymc_summary_df.columns:
    pymc_summary_df[col] = pd.to_numeric(pymc_summary_df[col], errors='coerce')

pymc_summary_dict = pymc_summary_df.to_dict('index')

# Convert summary to serialisable format
pymc_coefficients = {}
for param, vals in pymc_summary_dict.items():
    pymc_coefficients[str(param)] = {
        k: float(v) if not pd.isna(v) else None for k, v in vals.items()
    }

log_entry = {
    'description': 'Bayesian analysis of AI safety behaviour under pressure',
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'methods': {
        'conjugate': 'Beta-Binomial with Beta(1,1) prior',
        'regression': 'Bayesian logistic regression (PyMC)',
        'mcmc': {
            'draws': 2000,
            'tune': 1000,
            'chains': 2,
            'target_accept': 0.9,
            'max_rhat': float(rhat_max),
            'min_ess_bulk': float(ess_bulk_min),
        }
    },
    'pressure_posteriors': [
        {k: v for k, v in s.items()}
        for s in pressure_stats
    ],
    'pressure_contrasts': contrasts,
    'monitoring_effect': {
        'prob_monitored_safer': float(prob_mon_safer),
        'mean_delta': float(delta_mon.mean()),
        'cri_delta': [float(delta_mon_cri[0]), float(delta_mon_cri[1])],
        'conclusion': 'Strong' if prob_mon_safer > 0.95 else 'Weak' if prob_mon_safer > 0.80 else 'Inconclusive'
    },
    'scenario_posteriors': [
        {k: v for k, v in s.items()}
        for s in scenario_stats
    ],
    'pymc_regression_coefficients': pymc_coefficients,
}

results_path = RESULTS_DIR / 'bayesian/metrics.json'
with open(results_path, 'w') as f:
    json.dump(log_entry, f, indent=2, default=str)

print(f'Results saved to: {results_path}')

## Summary

**Key Bayesian findings:**

- **Pressure effects:** Credible intervals for all pressure conditions heavily overlap with the baseline, providing no strong evidence that any single pressure condition systematically increases the unsafe rate beyond what could arise from sampling variability.
- **Monitoring effect:** The posterior distributions for monitored vs unmonitored conditions are nearly identical, consistent with the frequentist null result. The Bayesian analysis quantifies this: the 95% CrI for the monitoring difference straddles zero.
- **Scenario vulnerability:** `eligibility_triage` is clearly the most vulnerable scenario, with posterior mass well above the 20% unsafe threshold. Other scenarios have posterior mass concentrated near zero.
- **Bayesian logistic regression:** The scenario effect dominates; pressure and monitoring coefficients are centred near zero with wide credible intervals — consistent with the hypothesis that task design (scenario) is the primary driver of unsafe behaviour, not adversarial pressure or oversight.

**Implications for scalable oversight:**
- The Bayesian framework provides honest uncertainty quantification — we can say "there is X% probability that pressure P increases unsafe rate" rather than relying on binary significance tests.
- The strong scenario effect suggests that oversight resources should be allocated based on **task type** rather than assumed adversarial conditions.
- The null monitoring result, robustly confirmed by Bayesian analysis, challenges the assumption that monitoring disclosure alone is sufficient for safety.